# Coordinated Electricity Demand and Wind Power Forecasting

An end-to-end, reproducible machine-learning workflow for short-term forecasting of electricity demand and wind power. The two targets are predicted by separate models whose outputs are then coordinated downstream into a residual-load estimate and a rule-based operational decision-support layer.

The notebook covers:

- Time-ordered train/test splitting and rolling-origin (walk-forward) validation
- A panel of baseline and ensemble models (mean baseline, Linear, Ridge, Random Forest, Gradient Boosting, Extra Trees, and XGBoost)
- Diagnostic figures (actual vs. predicted time series and scatter, residuals, error-band distribution)
- Model interpretability with global feature importance, SHAP, and local LIME explanations
- A transparent, scenario-based residual-load estimate that couples the two predictors
- A rule-based operational layer with illustrative KPIs and a safe LLM prompt template

## Datasets

Both datasets are publicly available on Kaggle:

- Electricity load: https://www.kaggle.com/datasets/saurabhshahane/electricity-load-forecasting/data
- Wind power: https://www.kaggle.com/datasets/mubashirrahim/wind-power-generation-data-forecasting/data

## Usage

The notebook runs locally or on Google Colab. By default it reads datasets from `./data` and writes figures, tables, and trained models to `./outputs`. Edit the paths in the configuration cell to match your environment (for example, a Google Drive folder when running on Colab).


In [ ]:
# ============================================================
# 0. Environment setup
# ============================================================
# Runs locally or on Google Colab. Configure DATA_DIR and OUTPUT_DIR below to
# match your environment. On Colab you can optionally mount Google Drive and
# point these paths to a Drive folder.

import os
import sys
import json
import math
import time
import warnings
from pathlib import Path
from datetime import datetime

warnings.filterwarnings("ignore")

IN_COLAB = "google.colab" in sys.modules

# Optional: mount Google Drive on Colab (set MOUNT_DRIVE = True to enable).
MOUNT_DRIVE = False
if IN_COLAB and MOUNT_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')

# Input data and output locations (edit as needed).
DATA_DIR = Path('data')
OUTPUT_DIR = Path('outputs')
FIG_DIR = OUTPUT_DIR / 'figures'
TABLE_DIR = OUTPUT_DIR / 'tables'
MODEL_DIR = OUTPUT_DIR / 'models'
for p in [OUTPUT_DIR, FIG_DIR, TABLE_DIR, MODEL_DIR]:
    p.mkdir(parents=True, exist_ok=True)

RANDOM_STATE = 10
TEST_SIZE = 0.30
WIND_FARM_CAPACITY_MW = 1000.0

print(f"Data directory  : {DATA_DIR.resolve()}")
print(f"Output directory: {OUTPUT_DIR.resolve()}")


In [ ]:
# ============================================================
# 1. Optional package installation
# ============================================================
# Run this cell in Colab if packages are missing. It is intentionally safe:
# installed packages are skipped when already available.

import importlib.util
import subprocess

def ensure_package(import_name, pip_name=None):
    if importlib.util.find_spec(import_name) is None:
        pkg = pip_name or import_name
        print(f"Installing {pkg} ...")
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pkg])

for import_name, pip_name in [
    ('xgboost', 'xgboost'),
    ('shap', 'shap'),
    ('lime', 'lime'),
    ('joblib', 'joblib'),
]:
    ensure_package(import_name, pip_name)

print('Package check complete.')


In [ ]:
# ============================================================
# 2. Imports and reusable utilities
# ============================================================
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib

from sklearn.ensemble import ExtraTreesRegressor, RandomForestRegressor, GradientBoostingRegressor
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.dummy import DummyRegressor
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error, median_absolute_error, max_error
from sklearn.inspection import permutation_importance
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

try:
    from xgboost import XGBRegressor
    HAS_XGB = True
except Exception as e:
    HAS_XGB = False
    print('XGBoost unavailable:', e)

try:
    import shap
    HAS_SHAP = True
except Exception as e:
    HAS_SHAP = False
    print('SHAP unavailable:', e)

try:
    from lime.lime_tabular import LimeTabularExplainer
    HAS_LIME = True
except Exception as e:
    HAS_LIME = False
    print('LIME unavailable:', e)

np.random.seed(RANDOM_STATE)

plt.rcParams.update({
    'font.size': 14,
    'axes.titlesize': 15,
    'axes.labelsize': 14,
    'xtick.labelsize': 12,
    'ytick.labelsize': 12,
    'legend.fontsize': 12,
    'figure.titlesize': 16,
})

def save_fig(path, dpi=400):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    plt.tight_layout()
    plt.savefig(path, dpi=dpi, bbox_inches='tight')
    plt.savefig(path.with_suffix('.pdf'), bbox_inches='tight')
    print(f'Saved: {path}')


def regression_metrics(y_true, y_pred, prefix=None):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    mse = mean_squared_error(y_true, y_pred)
    out = {
        'R2': r2_score(y_true, y_pred),
        'MSE': mse,
        'RMSE': math.sqrt(mse),
        'MAE': mean_absolute_error(y_true, y_pred),
        'MedianAE': median_absolute_error(y_true, y_pred),
        'MaxError': max_error(y_true, y_pred),
    }
    denom = np.where(np.abs(y_true) < 1e-8, np.nan, np.abs(y_true))
    mape = np.nanmean(np.abs((y_true - y_pred) / denom))
    out['MAPE'] = mape
    if prefix:
        out = {f'{prefix}_{k}': v for k, v in out.items()}
    return out


def print_and_save_table(df, name):
    display(df)
    csv_path = TABLE_DIR / f'{name}.csv'
    xlsx_path = TABLE_DIR / f'{name}.xlsx'
    df.to_csv(csv_path, index=False)
    df.to_excel(xlsx_path, index=False)
    print(f'Saved: {csv_path}')
    print(f'Saved: {xlsx_path}')


def time_ordered_split(X, y, test_size=0.30):
    n = len(X)
    split_idx = int(n * (1 - test_size))
    return X.iloc[:split_idx].copy(), X.iloc[split_idx:].copy(), y.iloc[:split_idx].copy(), y.iloc[split_idx:].copy()


## 3. Dataset acquisition

Two options are supported.

1. If the datasets already exist locally, place them under `DATA_DIR` and keep the default paths.
2. If they are not present, run the Kaggle-download cell after configuring your Kaggle credentials (`kaggle.json`).


In [ ]:
# ============================================================
# 3.1 Dataset paths
# ============================================================
ELEC_PATH = DATA_DIR / 'ElecLoadForcasting' / 'dataset.csv'
WIND_PATH = DATA_DIR / 'WindPowerData' / 'dataset.csv'

print('Electricity path:', ELEC_PATH)
print('Wind path       :', WIND_PATH)
print('Electricity file exists:', ELEC_PATH.exists())
print('Wind file exists       :', WIND_PATH.exists())


In [ ]:
# ============================================================
# 3.2 Optional Kaggle download helper
# ============================================================
# Run this cell only when the datasets are missing. Configure Kaggle credentials
# first: place kaggle.json under ~/.kaggle/kaggle.json (or /content/kaggle.json on Colab).

DOWNLOAD_FROM_KAGGLE = False  # Change to True when needed.

if DOWNLOAD_FROM_KAGGLE:
    ensure_package('kaggle', 'kaggle')
    import shutil
    kaggle_json_candidates = [Path('/content/kaggle.json'), Path.home()/'.kaggle'/'kaggle.json']
    for candidate in kaggle_json_candidates:
        if candidate.exists():
            kaggle_dir = Path.home()/'.kaggle'
            kaggle_dir.mkdir(exist_ok=True)
            shutil.copy(candidate, kaggle_dir/'kaggle.json')
            os.chmod(kaggle_dir/'kaggle.json', 0o600)
            break
    else:
        raise FileNotFoundError('kaggle.json was not found. Provide it under ~/.kaggle/kaggle.json (or /content/kaggle.json on Colab).')

    import zipfile
    import subprocess
    elec_dir = DATA_DIR / 'ElecLoadForcasting'
    wind_dir = DATA_DIR / 'WindPowerData'
    elec_dir.mkdir(parents=True, exist_ok=True)
    wind_dir.mkdir(parents=True, exist_ok=True)

    subprocess.check_call(['kaggle', 'datasets', 'download', '-d', 'saurabhshahane/electricity-load-forecasting', '-p', str(elec_dir), '--unzip'])
    subprocess.check_call(['kaggle', 'datasets', 'download', '-d', 'mubashirrahim/wind-power-generation-data-forecasting', '-p', str(wind_dir), '--unzip'])

    print('Download complete.')
    print(list(elec_dir.glob('*')))
    print(list(wind_dir.glob('*')))


In [ ]:
# ============================================================
# 4. Load and preprocess datasets
# ============================================================
if not ELEC_PATH.exists():
    raise FileNotFoundError(f'Electricity dataset not found: {ELEC_PATH}')
if not WIND_PATH.exists():
    raise FileNotFoundError(f'Wind dataset not found: {WIND_PATH}')

raw_elec = pd.read_csv(ELEC_PATH)
raw_wind = pd.read_csv(WIND_PATH)

print('Raw electricity shape:', raw_elec.shape)
print('Raw wind shape       :', raw_wind.shape)
print('\nElectricity columns:', list(raw_elec.columns))
print('\nWind columns:', list(raw_wind.columns))


def prepare_electricity(df):
    df = df.copy()
    if 'datetime' not in df.columns:
        raise ValueError('Electricity dataset must contain a datetime column.')
    df['datetime'] = pd.to_datetime(df['datetime'])
    df = df.sort_values('datetime').reset_index(drop=True)
    df['Hour'] = df['datetime'].dt.hour
    df['Month'] = df['datetime'].dt.month
    drop_cols = [c for c in ['Holiday_ID', 'school'] if c in df.columns]
    df = df.drop(columns=drop_cols)
    target = 'nat_demand'
    if target not in df.columns:
        raise ValueError('Electricity target nat_demand not found.')
    time_col = df['datetime'].copy()
    X = df.drop(columns=[target, 'datetime'])
    y = df[target]
    # keep numerical columns only for tree and baseline models
    X = X.select_dtypes(include=[np.number]).copy()
    return X, y, time_col, df


def prepare_wind(df):
    df = df.copy()
    time_candidates = ['Time', 'time', 'datetime', 'DateTime', 'date']
    time_col_name = next((c for c in time_candidates if c in df.columns), None)
    if time_col_name is None:
        raise ValueError('Wind dataset must contain a time column, e.g., Time.')
    df[time_col_name] = pd.to_datetime(df[time_col_name])
    df = df.sort_values(time_col_name).reset_index(drop=True)
    df['Hour'] = df[time_col_name].dt.hour
    df['Month'] = df[time_col_name].dt.month
    target = 'Power'
    if target not in df.columns:
        raise ValueError('Wind target Power not found.')
    time_col = df[time_col_name].copy()
    X = df.drop(columns=[target, time_col_name])
    y = df[target]
    X = X.select_dtypes(include=[np.number]).copy()
    return X, y, time_col, df

X_elec, y_elec, t_elec, elec_df = prepare_electricity(raw_elec)
X_wind, y_wind, t_wind, wind_df = prepare_wind(raw_wind)

# Missing value handling: median imputation by column, after reporting missingness.
missing_report = pd.DataFrame({
    'dataset': ['electricity']*len(X_elec.columns) + ['wind']*len(X_wind.columns),
    'feature': list(X_elec.columns) + list(X_wind.columns),
    'missing_count': list(X_elec.isna().sum()) + list(X_wind.isna().sum())
})
print_and_save_table(missing_report, 'missing_value_report')

X_elec = X_elec.fillna(X_elec.median(numeric_only=True))
X_wind = X_wind.fillna(X_wind.median(numeric_only=True))

print('Prepared electricity:', X_elec.shape, y_elec.shape)
print('Prepared wind       :', X_wind.shape, y_wind.shape)


In [ ]:
# ============================================================
# 5. Time-ordered train/test split
# ============================================================
X_elec_train, X_elec_test, y_elec_train, y_elec_test = time_ordered_split(X_elec, y_elec, TEST_SIZE)
X_wind_train, X_wind_test, y_wind_train, y_wind_test = time_ordered_split(X_wind, y_wind, TEST_SIZE)

data_split_summary = pd.DataFrame([
    {'Dataset': 'Electricity demand', 'Total': len(X_elec), 'Train': len(X_elec_train), 'Test': len(X_elec_test),
     'Train start': str(t_elec.iloc[0]), 'Train end': str(t_elec.iloc[len(X_elec_train)-1]),
     'Test start': str(t_elec.iloc[len(X_elec_train)]), 'Test end': str(t_elec.iloc[-1])},
    {'Dataset': 'Wind power', 'Total': len(X_wind), 'Train': len(X_wind_train), 'Test': len(X_wind_test),
     'Train start': str(t_wind.iloc[0]), 'Train end': str(t_wind.iloc[len(X_wind_train)-1]),
     'Test start': str(t_wind.iloc[len(X_wind_train)]), 'Test end': str(t_wind.iloc[-1])},
])
print_and_save_table(data_split_summary, 'data_split_summary')


In [ ]:
# ============================================================
# 6. Baseline and ExtraTrees models
# ============================================================
def make_models():
    models = {
        'Persistence/NaiveMean': DummyRegressor(strategy='mean'),
        'LinearRegression': LinearRegression(),
        'Ridge': Pipeline([('scaler', StandardScaler()), ('model', Ridge(alpha=1.0, random_state=RANDOM_STATE))]),
        'RandomForest': RandomForestRegressor(n_estimators=200, random_state=RANDOM_STATE, n_jobs=-1, min_samples_leaf=1),
        'GradientBoosting': GradientBoostingRegressor(random_state=RANDOM_STATE),
        'ExtraTrees': ExtraTreesRegressor(n_estimators=300, random_state=RANDOM_STATE, n_jobs=-1, min_samples_leaf=1),
    }
    if HAS_XGB:
        models['XGBoost'] = XGBRegressor(
            n_estimators=400, max_depth=6, learning_rate=0.05,
            subsample=0.9, colsample_bytree=0.9, objective='reg:squarederror',
            random_state=RANDOM_STATE, n_jobs=-1
        )
    return models


def evaluate_model_family(X_train, y_train, X_test, y_test, dataset_name):
    rows = []
    fitted = {}
    for name, model in make_models().items():
        start = time.time()
        model.fit(X_train, y_train)
        fit_time = time.time() - start
        pred = model.predict(X_test)
        metrics = regression_metrics(y_test, pred)
        metrics.update({'Dataset': dataset_name, 'Model': name, 'FitTimeSeconds': fit_time})
        rows.append(metrics)
        fitted[name] = model
        print(f'{dataset_name} | {name}: R2={metrics["R2"]:.4f}, RMSE={metrics["RMSE"]:.4f}, MAE={metrics["MAE"]:.4f}')
    df = pd.DataFrame(rows)
    cols = ['Dataset','Model','R2','RMSE','MAE','MSE','MAPE','MedianAE','MaxError','FitTimeSeconds']
    return df[cols], fitted

metrics_elec, models_elec = evaluate_model_family(X_elec_train, y_elec_train, X_elec_test, y_elec_test, 'Electricity demand')
metrics_wind, models_wind = evaluate_model_family(X_wind_train, y_wind_train, X_wind_test, y_wind_test, 'Wind power')

baseline_metrics = pd.concat([metrics_elec, metrics_wind], ignore_index=True)
print_and_save_table(baseline_metrics, 'baseline_comparison_time_ordered')

best_elec_model = models_elec['ExtraTrees']
best_wind_model = models_wind['ExtraTrees']
joblib.dump(best_elec_model, MODEL_DIR / 'electricity_extratrees_revision.pkl')
joblib.dump(best_wind_model, MODEL_DIR / 'wind_extratrees_revision.pkl')
print('Saved ExtraTrees models.')


In [ ]:
# ============================================================
# 7. Rolling-origin validation
# ============================================================
def rolling_origin_validation(X, y, dataset_name, model_factory=None, n_splits=5, initial_train_ratio=0.50):
    if model_factory is None:
        model_factory = lambda: ExtraTreesRegressor(n_estimators=200, random_state=RANDOM_STATE, n_jobs=-1)
    n = len(X)
    initial = int(n * initial_train_ratio)
    remaining = n - initial
    fold_size = remaining // n_splits
    rows = []
    for fold in range(n_splits):
        train_end = initial + fold * fold_size
        test_start = train_end
        test_end = initial + (fold + 1) * fold_size if fold < n_splits - 1 else n
        if test_end <= test_start:
            continue
        X_tr, y_tr = X.iloc[:train_end], y.iloc[:train_end]
        X_te, y_te = X.iloc[test_start:test_end], y.iloc[test_start:test_end]
        model = model_factory()
        model.fit(X_tr, y_tr)
        pred = model.predict(X_te)
        m = regression_metrics(y_te, pred)
        m.update({'Dataset': dataset_name, 'Fold': fold+1, 'TrainSize': len(X_tr), 'TestSize': len(X_te),
                  'TestStartIndex': test_start, 'TestEndIndex': test_end-1})
        rows.append(m)
    return pd.DataFrame(rows)

roll_elec = rolling_origin_validation(X_elec, y_elec, 'Electricity demand')
roll_wind = rolling_origin_validation(X_wind, y_wind, 'Wind power')
rolling_metrics = pd.concat([roll_elec, roll_wind], ignore_index=True)
print_and_save_table(rolling_metrics, 'rolling_origin_validation')

rolling_summary = rolling_metrics.groupby('Dataset')[['R2','RMSE','MAE','MAPE']].agg(['mean','std']).reset_index()
rolling_summary.columns = ['_'.join([str(a), str(b)]).strip('_') for a,b in rolling_summary.columns]
print_and_save_table(rolling_summary, 'rolling_origin_validation_summary')


In [ ]:
# ============================================================
# 8. Main predictions and diagnostic figures
# ============================================================
y_elec_pred = best_elec_model.predict(X_elec_test)
y_wind_pred = best_wind_model.predict(X_wind_test)

# Actual vs predicted, first and last 300 samples.
def plot_first_last(y_true, y_pred, dataset_name, ylabel, filename_prefix):
    for label, idx in [('first_300', np.arange(min(300, len(y_true)))), ('last_300', np.arange(max(0, len(y_true)-300), len(y_true)) )]:
        plt.figure(figsize=(11, 5))
        plt.plot(np.arange(len(idx)), np.asarray(y_true)[idx], label='Actual', linewidth=1.7)
        plt.plot(np.arange(len(idx)), np.asarray(y_pred)[idx], label='Predicted', linewidth=1.7, linestyle='--')
        plt.xlabel('Sample index within selected test window')
        plt.ylabel(ylabel)
        plt.title(f'{dataset_name}: Actual vs Predicted ({label.replace("_", " ")})')
        plt.legend()
        plt.grid(True, alpha=0.3)
        save_fig(FIG_DIR / f'{filename_prefix}_{label}.png')
        plt.show()

plot_first_last(y_wind_test.values, y_wind_pred, 'Wind power', 'Normalized wind power', 'figure_actual_predicted_wind')
plot_first_last(y_elec_test.values, y_elec_pred, 'Electricity demand', 'Demand (MW)', 'figure_actual_predicted_electricity')

# Scatter actual vs predicted.
def plot_scatter(y_true, y_pred, dataset_name, xlabel, ylabel, filename):
    plt.figure(figsize=(7, 6))
    plt.scatter(y_true, y_pred, s=12, alpha=0.45)
    lo = min(np.min(y_true), np.min(y_pred))
    hi = max(np.max(y_true), np.max(y_pred))
    plt.plot([lo, hi], [lo, hi], linestyle='--', linewidth=2)
    plt.xlabel(xlabel)
    plt.ylabel(ylabel)
    plt.title(f'{dataset_name}: Actual vs Predicted')
    plt.grid(True, alpha=0.3)
    save_fig(FIG_DIR / filename)
    plt.show()

plot_scatter(y_wind_test.values, y_wind_pred, 'Wind power', 'Actual normalized power', 'Predicted normalized power', 'figure_scatter_wind.png')
plot_scatter(y_elec_test.values, y_elec_pred, 'Electricity demand', 'Actual demand (MW)', 'Predicted demand (MW)', 'figure_scatter_electricity.png')


In [ ]:
# ============================================================
# 9. Error bands and residual diagnostics
# ============================================================
def error_band_table(y_true, y_pred, dataset_name):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    pct = np.abs(y_true - y_pred) / np.maximum(np.abs(y_true), 1e-8) * 100
    bins = [0, 5, 10, 20, 40, np.inf]
    labels = ['<=5%', '5-10%', '10-20%', '20-40%', '>40%']
    cat = pd.cut(pct, bins=bins, labels=labels, include_lowest=True, right=True)
    counts = cat.value_counts().reindex(labels, fill_value=0)
    return pd.DataFrame({'Dataset': dataset_name, 'Error band': labels, 'Count': counts.values,
                         'Percentage': counts.values / len(pct) * 100})

err_elec = error_band_table(y_elec_test.values, y_elec_pred, 'Electricity demand')
err_wind = error_band_table(y_wind_test.values, y_wind_pred, 'Wind power')
error_bands = pd.concat([err_elec, err_wind], ignore_index=True)
print_and_save_table(error_bands, 'error_band_distribution')

for dataset, y_true, y_pred, fname in [
    ('Wind power', y_wind_test.values, y_wind_pred, 'figure_residuals_wind.png'),
    ('Electricity demand', y_elec_test.values, y_elec_pred, 'figure_residuals_electricity.png'),
]:
    residuals = np.asarray(y_true) - np.asarray(y_pred)
    plt.figure(figsize=(7, 6))
    plt.scatter(y_pred, residuals, s=12, alpha=0.45)
    plt.axhline(0, linestyle='--', linewidth=2)
    plt.xlabel('Predicted value')
    plt.ylabel('Residual (actual - predicted)')
    plt.title(f'{dataset}: Residuals vs Predicted')
    plt.grid(True, alpha=0.3)
    save_fig(FIG_DIR / fname)
    plt.show()

    plt.figure(figsize=(7, 5))
    plt.hist(residuals, bins=50, alpha=0.85)
    plt.xlabel('Residual (actual - predicted)')
    plt.ylabel('Frequency')
    plt.title(f'{dataset}: Residual distribution')
    plt.grid(True, alpha=0.3)
    save_fig(FIG_DIR / fname.replace('residuals', 'residual_distribution'))
    plt.show()


In [ ]:
# ============================================================
# 10. Feature mapping and residual-load scenario variable
# ============================================================
# The two public datasets are not co-located. Therefore, a directly measured residual-load
# target is unavailable. This cell produces a transparent scenario-based residual-load
# estimate using demand predictions and mapped wind predictions.

def map_electricity_to_wind_features(elec_X, wind_feature_columns, alpha=0.143):
    rows = []
    for _, row in elec_X.iterrows():
        # Use available demand-side variables. The selected formulas are explicit and reproducible.
        t2m = row.get('T2M_toc', np.nan)
        qv = row.get('QV2M_toc', np.nan)
        w2m = row.get('W2M_toc', np.nan)
        hour = row.get('Hour', np.nan)
        month = row.get('Month', np.nan)

        # humidity fraction to percentage where needed
        rh_pct = qv * 100 if pd.notna(qv) else np.nan
        # approximate dew point: simple operational proxy used only for scenario mapping
        dewpoint = t2m - ((100 - rh_pct) / 5.0) if pd.notna(t2m) and pd.notna(rh_pct) else t2m - 2
        # power-law profile from 2 m to 10 m and 100 m
        wind10 = w2m * ((10.0 / 2.0) ** alpha) if pd.notna(w2m) else np.nan
        wind100 = w2m * ((100.0 / 2.0) ** alpha) if pd.notna(w2m) else np.nan
        gust10 = wind10 * 1.30 if pd.notna(wind10) else np.nan

        base = {
            'temperature_2m': t2m,
            'relativehumidity_2m': rh_pct,
            'dewpoint_2m': dewpoint,
            'windspeed_10m': wind10,
            'windspeed_100m': wind100,
            'winddirection_10m': 180.0,
            'winddirection_100m': 180.0,
            'windgusts_10m': gust10,
            'Hour': hour,
            'Month': month,
        }
        rows.append({c: base.get(c, 0.0) for c in wind_feature_columns})
    return pd.DataFrame(rows, columns=wind_feature_columns).fillna(0.0)

mapped_wind_X_from_elec_test = map_electricity_to_wind_features(X_elec_test, X_wind.columns)
scenario_wind_fraction = np.clip(best_wind_model.predict(mapped_wind_X_from_elec_test), 0, 1)
scenario_wind_mw = scenario_wind_fraction * WIND_FARM_CAPACITY_MW
scenario_demand_mw = y_elec_pred
scenario_residual_mw = np.maximum(scenario_demand_mw - scenario_wind_mw, 0)

residual_scenario = pd.DataFrame({
    'predicted_demand_MW': scenario_demand_mw,
    'mapped_predicted_wind_fraction': scenario_wind_fraction,
    'mapped_predicted_wind_MW': scenario_wind_mw,
    'scenario_residual_load_MW': scenario_residual_mw,
    'wind_share_of_predicted_demand': np.divide(scenario_wind_mw, np.maximum(scenario_demand_mw, 1e-8)),
})

summary_residual = residual_scenario.describe().T.reset_index().rename(columns={'index': 'Variable'})
print_and_save_table(summary_residual, 'residual_load_scenario_summary')
residual_scenario.head(1000).to_csv(TABLE_DIR / 'residual_load_scenario_first1000.csv', index=False)

plt.figure(figsize=(11,5))
N = min(300, len(residual_scenario))
plt.plot(residual_scenario['predicted_demand_MW'].iloc[:N].values, label='Predicted demand')
plt.plot(residual_scenario['mapped_predicted_wind_MW'].iloc[:N].values, label='Mapped wind generation')
plt.plot(residual_scenario['scenario_residual_load_MW'].iloc[:N].values, label='Scenario residual load')
plt.xlabel('Test sample index')
plt.ylabel('Power (MW)')
plt.title('Scenario-based residual-load computation from coordinated predictors')
plt.legend()
plt.grid(True, alpha=0.3)
save_fig(FIG_DIR / 'figure_residual_load_scenario.png')
plt.show()


In [ ]:
# ============================================================
# 11. Global feature importance and SHAP analysis
# ============================================================
def plot_feature_importance(model, X, dataset_name, filename, top_n=12):
    if hasattr(model, 'feature_importances_'):
        imp = pd.DataFrame({'Feature': X.columns, 'Importance': model.feature_importances_})
    else:
        result = permutation_importance(model, X, y_elec_test if 'Electricity' in dataset_name else y_wind_test, random_state=RANDOM_STATE, n_repeats=5)
        imp = pd.DataFrame({'Feature': X.columns, 'Importance': result.importances_mean})
    imp = imp.sort_values('Importance', ascending=False).head(top_n)
    plt.figure(figsize=(9, 6))
    plt.barh(imp['Feature'][::-1], imp['Importance'][::-1])
    plt.xlabel('Importance')
    plt.title(f'{dataset_name}: global feature importance')
    plt.grid(True, axis='x', alpha=0.3)
    save_fig(FIG_DIR / filename)
    plt.show()
    return imp

imp_elec = plot_feature_importance(best_elec_model, X_elec_train, 'Electricity demand', 'figure_global_importance_electricity.png')
imp_wind = plot_feature_importance(best_wind_model, X_wind_train, 'Wind power', 'figure_global_importance_wind.png')
print_and_save_table(imp_elec.assign(Dataset='Electricity demand'), 'global_importance_electricity')
print_and_save_table(imp_wind.assign(Dataset='Wind power'), 'global_importance_wind')

if HAS_SHAP:
    for dataset_name, model, X_test, fname in [
        ('Electricity demand', best_elec_model, X_elec_test, 'figure_shap_summary_electricity.png'),
        ('Wind power', best_wind_model, X_wind_test, 'figure_shap_summary_wind.png'),
    ]:
        sample = X_test.sample(n=min(1000, len(X_test)), random_state=RANDOM_STATE)
        explainer = shap.TreeExplainer(model)
        shap_values = explainer.shap_values(sample)
        plt.figure(figsize=(10, 7))
        shap.summary_plot(shap_values, sample, show=False, max_display=12)
        plt.title(f'{dataset_name}: TreeSHAP summary')
        save_fig(FIG_DIR / fname)
        plt.show()
else:
    print('SHAP not available; global feature importance has been generated instead.')


In [ ]:
# ============================================================
# 12. Local explanations with LIME
# ============================================================
def generate_lime_plot(model, X_train, X_test, y_test, instance_index, dataset_name, filename, target_unit):
    if not HAS_LIME:
        print('LIME is not available. Skipping LIME plot.')
        return None
    instance_index = min(instance_index, len(X_test)-1)
    explainer = LimeTabularExplainer(
        training_data=np.asarray(X_train),
        feature_names=list(X_train.columns),
        mode='regression',
        discretize_continuous=True,
        random_state=RANDOM_STATE
    )
    x = X_test.iloc[instance_index]
    exp = explainer.explain_instance(
        data_row=x.values,
        predict_fn=model.predict,
        num_features=min(10, X_train.shape[1])
    )
    pred = float(model.predict(X_test.iloc[[instance_index]])[0])
    actual = float(y_test.iloc[instance_index])
    fig = exp.as_pyplot_figure(label=1)
    fig.set_size_inches(10, 7)
    plt.title(f'{dataset_name} LIME explanation\nRecord {instance_index}: actual={actual:.4f} {target_unit}, predicted={pred:.4f} {target_unit}', fontsize=15)
    plt.xlabel('Local contribution', fontsize=14)
    plt.xticks(fontsize=12)
    plt.yticks(fontsize=12)
    save_fig(FIG_DIR / filename, dpi=500)
    plt.show()
    return {'Dataset': dataset_name, 'Record': instance_index, 'Actual': actual, 'Predicted': pred, 'Figure': filename}

lime_rows = []
lime_rows.append(generate_lime_plot(best_wind_model, X_wind_train, X_wind_test, y_wind_test, 7500, 'Wind power', 'figure_lime_wind_record_7500.png', 'fraction'))
lime_rows.append(generate_lime_plot(best_wind_model, X_wind_train, X_wind_test, y_wind_test, 13000, 'Wind power', 'figure_lime_wind_record_13000.png', 'fraction'))
lime_rows.append(generate_lime_plot(best_elec_model, X_elec_train, X_elec_test, y_elec_test, 5000, 'Electricity demand', 'figure_lime_electricity_record_5000.png', 'MW'))
lime_rows.append(generate_lime_plot(best_elec_model, X_elec_train, X_elec_test, y_elec_test, 10000, 'Electricity demand', 'figure_lime_electricity_record_10000.png', 'MW'))
lime_summary = pd.DataFrame([r for r in lime_rows if r is not None])
if not lime_summary.empty:
    print_and_save_table(lime_summary, 'lime_instance_summary')


In [ ]:
# ============================================================
# 13. Rule-based operational layer and KPI scenario evaluation
# ============================================================
def agentic_energy_ai(predicted_demand_mw, wind_mw, wind_farm_capacity_mw=WIND_FARM_CAPACITY_MW):
    grid_mw = max(predicted_demand_mw - wind_mw, 0)
    max_allowed_grid = 0.8 * predicted_demand_mw
    actions = []
    if wind_mw < 0.20 * wind_farm_capacity_mw:
        actions.append('Discharge battery storage to support low renewable generation')
    if predicted_demand_mw > 1500:
        actions.append('Adjust smart HVAC schedule to reduce peak demand')
    if grid_mw > 800:
        actions.append('Shift EV charging to off-peak period')
    if wind_mw >= 0.50 * wind_farm_capacity_mw:
        actions.append('Store excess wind generation to avoid curtailment')
    if predicted_demand_mw < 500:
        actions.append('Activate energy-saving mode for non-critical loads')
    if grid_mw > max_allowed_grid:
        actions.append('Issue carbon-alert and activate flexible demand response')
    if not actions:
        actions.append('Normal operation with monitoring')
    return {'grid_MW': grid_mw, 'actions': actions, 'max_allowed_grid_MW': max_allowed_grid}

# Simple scenario KPI assumptions for decision-support evaluation.
# These are not power-flow results; they quantify the rule-layer effect under transparent assumptions.
FLEXIBLE_LOAD_CAPACITY_MW = 100.0
BATTERY_POWER_LIMIT_MW = 200.0
EMISSION_FACTOR_TCO2_PER_MWH = 0.45

baseline_grid = residual_scenario['scenario_residual_load_MW'].values
wind_mw = residual_scenario['mapped_predicted_wind_MW'].values
demand_mw = residual_scenario['predicted_demand_MW'].values

controlled_grid = []
action_counts = {}
for d, w, g in zip(demand_mw, wind_mw, baseline_grid):
    out = agentic_energy_ai(d, w)
    reduction = 0.0
    if 'Discharge battery storage to support low renewable generation' in out['actions']:
        reduction += min(BATTERY_POWER_LIMIT_MW, g)
    if 'Adjust smart HVAC schedule to reduce peak demand' in out['actions']:
        reduction += min(FLEXIBLE_LOAD_CAPACITY_MW, max(g - reduction, 0))
    if 'Shift EV charging to off-peak period' in out['actions']:
        reduction += min(50.0, max(g - reduction, 0))
    if 'Issue carbon-alert and activate flexible demand response' in out['actions']:
        reduction += min(50.0, max(g - reduction, 0))
    controlled_grid.append(max(g - reduction, 0))
    for a in out['actions']:
        action_counts[a] = action_counts.get(a, 0) + 1
controlled_grid = np.asarray(controlled_grid)

kpi = pd.DataFrame([
    {'KPI': 'Mean grid draw before control (MW)', 'Value': np.mean(baseline_grid)},
    {'KPI': 'Mean grid draw after rule layer (MW)', 'Value': np.mean(controlled_grid)},
    {'KPI': 'Mean grid-draw reduction (MW)', 'Value': np.mean(baseline_grid - controlled_grid)},
    {'KPI': 'Peak grid draw before control (MW)', 'Value': np.max(baseline_grid)},
    {'KPI': 'Peak grid draw after rule layer (MW)', 'Value': np.max(controlled_grid)},
    {'KPI': 'Peak shaving (MW)', 'Value': np.max(baseline_grid) - np.max(controlled_grid)},
    {'KPI': 'Estimated avoided emissions per hour-equivalent (tCO2)', 'Value': np.sum(baseline_grid - controlled_grid) * EMISSION_FACTOR_TCO2_PER_MWH},
])
print_and_save_table(kpi, 'agentic_rule_layer_kpis')

action_df = pd.DataFrame([{'Action': k, 'Count': v, 'Percentage': v/len(baseline_grid)*100} for k,v in action_counts.items()])
print_and_save_table(action_df, 'agentic_action_distribution')

plt.figure(figsize=(11,5))
N = min(300, len(baseline_grid))
plt.plot(baseline_grid[:N], label='Before rule layer')
plt.plot(controlled_grid[:N], label='After rule layer', linestyle='--')
plt.xlabel('Test sample index')
plt.ylabel('Grid draw (MW)')
plt.title('Operational scenario: grid draw before and after rule-based decision layer')
plt.legend()
plt.grid(True, alpha=0.3)
save_fig(FIG_DIR / 'figure_agentic_kpi_grid_draw.png')
plt.show()


In [ ]:
# ============================================================
# 14. Safe LLM prompt template without API key exposure
# ============================================================
# This function builds the exact operator-facing prompt. It does not call an external LLM.
# The final system should pass the prompt to a controlled, logged LLM interface only after
# operator review and safety constraints.

def build_operator_prompt(predicted_demand_mw, wind_mw, grid_mw, actions, user_query=''):
    return f"""
You are an operator-facing energy-management reporting assistant.
Do not issue unsupported protection, relay, or safety-critical control instructions.
Use only the numerical forecasts and rule-based recommendations provided below.

Inputs:
- Predicted electricity demand: {predicted_demand_mw:.2f} MW
- Predicted wind generation: {wind_mw:.2f} MW
- Residual grid demand: {grid_mw:.2f} MW
- Rule-based actions: {actions}
- Operator query: {user_query}

Required output:
1. Situation summary.
2. Risk level: Low, Medium, or High, with reason.
3. Recommended operator-supervised actions.
4. Explanation of which forecast drivers contributed most.
5. Safety note: confirm that final execution requires operator authorization.
""".strip()

example = agentic_energy_ai(float(demand_mw[0]), float(wind_mw[0]))
example_prompt = build_operator_prompt(float(demand_mw[0]), float(wind_mw[0]), float(example['grid_MW']), example['actions'], 'How should the operator reduce grid draw?')
print(example_prompt)
with open(OUTPUT_DIR / 'llm_operator_prompt_template.txt', 'w', encoding='utf-8') as f:
    f.write(example_prompt)


In [ ]:
# ============================================================
# 15. Generate outputs summary
# ============================================================
summary_lines = []
summary_lines.append('Forecasting Workflow — Outputs Summary')
summary_lines.append('='*70)
summary_lines.append(f'Generated on: {datetime.now()}')
summary_lines.append('')
summary_lines.append('Datasets')
summary_lines.append(f'- Electricity load dataset: {ELEC_PATH}')
summary_lines.append(f'- Wind power dataset: {WIND_PATH}')
summary_lines.append('')
summary_lines.append('Time-ordered split summary')
summary_lines.append(data_split_summary.to_string(index=False))
summary_lines.append('')
summary_lines.append('Baseline comparison')
summary_lines.append(baseline_metrics.to_string(index=False))
summary_lines.append('')
summary_lines.append('Rolling-origin validation summary')
summary_lines.append(rolling_summary.to_string(index=False))
summary_lines.append('')
summary_lines.append('Residual-load scenario summary')
summary_lines.append(summary_residual.to_string(index=False))
summary_lines.append('')
summary_lines.append('Agentic rule-layer KPIs')
summary_lines.append(kpi.to_string(index=False))
summary_lines.append('')
summary_lines.append('Generated figures')
for fig in sorted(FIG_DIR.glob('*.png')):
    summary_lines.append(f'- {fig.name}')
summary_lines.append('')
summary_lines.append('Generated tables')
for tab in sorted(TABLE_DIR.glob('*.csv')):
    summary_lines.append(f'- {tab.name}')

summary_text = '\n'.join(summary_lines)
summary_path = OUTPUT_DIR / 'outputs_summary.txt'
with open(summary_path, 'w', encoding='utf-8') as f:
    f.write(summary_text)
print(summary_text[:5000])
print(f'\nSaved outputs summary to: {summary_path}')


## Summary of generated outputs

Running all cells produces a reproducible set of artifacts under `outputs/`:

- **Tables** (`outputs/tables/`): missing-value report, time-ordered data-split summary, baseline model comparison, rolling-origin validation results and summary, error-band distribution, global feature-importance rankings, residual-load scenario summary, rule-layer KPIs, and the action distribution.
- **Figures** (`outputs/figures/`): actual-vs-predicted time series and scatter plots, residual diagnostics, global feature importance, SHAP summaries, LIME local explanations, the residual-load scenario plot, and the operational grid-draw plot.
- **Models** (`outputs/models/`): the trained Extra Trees predictors for electricity demand and wind power.
- **Other**: a safe operator-facing LLM prompt template and a consolidated text summary.

Together these provide a transparent, reproducible baseline for coordinated electricity-demand and wind-power forecasting. The same structure can be adapted to other datasets and weather-driven resources (for example, photovoltaic generation) by adding a task-specific predictor with the relevant input features while reusing the validation, interpretability, residual-load, and reporting components.
